# Lab 03 — Reusable Skills

## Objective

Build a reusable research summary skill with a stable result schema.

## Introduction and concept

summarize follows the same topic filter, evidence limit and Summary schema for each source.

## Real-world use case

A course assistant extracts short grounded summaries from database and web notes.

## Technology stack and dependencies

Python dataclasses. Use Python 3.11 or newer and a Jupyter kernel from the environment where you installed the [assignment requirements](../requirements.txt). Start the notebook from its lab directory.

## Architecture and workflow

Skill contract → Evidence selection → Structured Summary


## Step 1 — Implement the components

summarize follows the same topic filter, evidence limit and Summary schema for each source.

In [1]:
import re
from dataclasses import asdict, dataclass

SKILL_INSTRUCTIONS = "Select up to two sentences relevant to the topic. Quote only the input."
MAX_QUOTED_SENTENCES = 2

@dataclass(frozen=True)
class Summary:
    topic: str
    source: str
    evidence: tuple[str, ...]

def summarize(topic, source, text):
    if not all(field.strip() for field in (topic, source, text)):
        raise ValueError("Topic, source ID and text are required")
    hits = [
        sentence.strip()
        for sentence in re.findall(r"[^.!?]+[.!?]?", text)
        if topic.casefold() in sentence.casefold()
    ]
    return Summary(topic, source, tuple(hits[:MAX_QUOTED_SENTENCES]))

## Step 2 — Run sample inputs and inspect actual output

Reuse the skill with database-index and web-cache passages. Each output follows the same schema and quotes its original source.

In [2]:
def main():
    cases = [
        ("index", "DB01", "An index can accelerate lookups. Writes can become slower."),
        ("cache", "WEB01", "A cache stores reused responses. A cache has an expiry."),
    ]
    print(SKILL_INSTRUCTIONS)
    for topic, source, text in cases:
        summary = summarize(topic, source, text)
        print(asdict(summary))
        assert 1 <= len(summary.evidence) <= 2
        assert all(quote in text for quote in summary.evidence)
    assert summarize("queue", "DB01", cases[0][2]).evidence == ()
    print("PASS: same schema and grounding rules for two domains")

main()

Select up to two sentences relevant to the topic. Quote only the input.
{'topic': 'index', 'source': 'DB01', 'evidence': ('An index can accelerate lookups.',)}
{'topic': 'cache', 'source': 'WEB01', 'evidence': ('A cache stores reused responses.', 'A cache has an expiry.')}
PASS: same schema and grounding rules for two domains


## Testing and validation

Reuse the skill with database-index and web-cache passages. Each output follows the same schema and quotes its original source. Assertions in the preceding cell check the demonstrated behavior and stop execution on a failed expectation.

## Limitations

The sentence splitter and matching are simple lexical rules; general-language summarization needs stronger segmentation and retrieval.

## How this relates to ExamPilot

Partial. ExamPilot separates reusable generation and validation modules, including backend/app/agents/question_generation_agent/question_generation_agent.py. The formal summary skill contract is a lab extension.

## Conclusion

The lab builds a reusable research summary skill with a stable result schema. Use the trace and assertions to explain which decisions the workflow makes.